# Controlled Semantic Activity-Mapping Evaluation

This notebook is a local backend-only evaluation version of the pipeline. It loads a fixed BPMN model and one synthetic event-log variant, runs the same preprocessing and semantic activity-mapping logic used by the backend, and stops after generating activity-mapping recommendations.

Its purpose is to support controlled validation of activity-label mismatches. For every mismatched log activity, it stores the full ranked list of BPMN candidates, component similarity scores, threshold decisions, and optional expected-vs-observed metrics.

**Set up the environment**:
   - Create a virtual environment named `.venv` in the current workspace.
   - Install the required dependencies listed in `requirements.txt`.
   - Set `bpmn_path`, `file_path`, and, when calculating validation metrics, `expected_mapping_path` for the synthetic version being evaluated. See `example-expected-activity-mapping.json` for the expected JSON structure.

In [1]:
# Verify validation pipeline configuration before importing project modules
from pathlib import Path

def resolve_activity_mapping_dir():
    notebook_filename = "activity-mapping-pipeline.ipynb"
    candidate_dirs = (
        Path.cwd(),
        Path.cwd() / "evaluation" / "activity-mapping",
        Path.cwd() / "activity-mapping",
        Path("platform") / "backend" / "evaluation" / "activity-mapping",
    )
    for candidate_dir in candidate_dirs:
        if (candidate_dir / notebook_filename).exists():
            return candidate_dir.resolve()
    raise RuntimeError("Configuration file not set: could not resolve activity-mapping notebook directory")

ACTIVITY_MAPPING_DIR = resolve_activity_mapping_dir()
VALIDATION_ENV_PATH = ACTIVITY_MAPPING_DIR / ".env"

_REQUIRED_CONFIG_FIELDS = (
    "EMBEDDING_MODEL",
    "LINGUISTIC_PIPELINE",
    "ACTION_WEIGHT",
    "OBJECT_WEIGHT",
    "FULL_LABEL_WEIGHT",
    "LEXICAL_WEIGHT",
    "TOKEN_WEIGHT",
    "MINIMUM_CANDIDATE_MARGIN",
    "CANDIDATE_THRESHOLD",
)

_config_values = {}
if not VALIDATION_ENV_PATH.exists():
    raise RuntimeError(f"Configuration file not set: validation .env not found at {VALIDATION_ENV_PATH}")

try:
    with VALIDATION_ENV_PATH.open(encoding="utf-8") as _env_file:
        for _line in _env_file:
            _line = _line.strip()
            if not _line or _line.startswith("#") or "=" not in _line:
                continue
            _key, _value = _line.split("=", 1)
            _config_values[_key.strip()] = _value.strip().strip('"').strip("'")
except OSError as exc:
    raise RuntimeError("Configuration file not set: validation .env could not be read") from exc

_missing_config = [
    _field for _field in _REQUIRED_CONFIG_FIELDS
    if not _config_values.get(_field)
]
if _missing_config:
    raise RuntimeError(
        "Configuration file not set: missing values for " + ", ".join(_missing_config)
    )

EMBEDDING_MODEL = _config_values["EMBEDDING_MODEL"]
LINGUISTIC_PIPELINE = _config_values["LINGUISTIC_PIPELINE"]
try:
    ACTION_WEIGHT = float(_config_values["ACTION_WEIGHT"])
    OBJECT_WEIGHT = float(_config_values["OBJECT_WEIGHT"])
    FULL_LABEL_WEIGHT = float(_config_values["FULL_LABEL_WEIGHT"])
    LEXICAL_WEIGHT = float(_config_values["LEXICAL_WEIGHT"])
    TOKEN_WEIGHT = float(_config_values["TOKEN_WEIGHT"])
    MINIMUM_CANDIDATE_MARGIN = float(_config_values["MINIMUM_CANDIDATE_MARGIN"])
    CANDIDATE_THRESHOLD = float(_config_values["CANDIDATE_THRESHOLD"])
except ValueError as exc:
    raise RuntimeError("Configuration file not set: numeric configuration values are invalid") from exc

_WEIGHT_TOTAL = ACTION_WEIGHT + OBJECT_WEIGHT + FULL_LABEL_WEIGHT + LEXICAL_WEIGHT + TOKEN_WEIGHT
if abs(_WEIGHT_TOTAL - 1.0) > 1e-6:
    raise RuntimeError(
        f"Configuration file not set: hybrid-score weights must sum to 1.0, got {_WEIGHT_TOTAL:.6f}"
    )

# ✅ Verify PM4Py installation and import
print ("🛠️ Importing Modules...")
import pm4py
from lxml import etree
import pandas
import re
import os
import io
import json
from datetime import datetime
# Keep notebook errors concise (no huge verbose tracebacks)
try:
    get_ipython().run_line_magic("xmode", "Minimal")
except Exception:
    pass

print ("✅ All modules imported!")


🛠️ Importing Modules...
Exception reporting mode: Minimal
✅ All modules imported!


In [2]:
# 🔇 Suppress known warnings from PM4Py and other libraries
import warnings
import os

# Suppress the ISO8601 datetime parsing warning from PM4Py
warnings.filterwarnings("ignore", message="ISO8601 strings are not fully supported with strpfromiso for Python versions below 3.11")

# Suppress HuggingFace symlinks warning
os.environ["HF_HUB_DISABLE_SYMLINKS_WARNING"] = "1"

# Suppress other common PM4Py warnings
warnings.filterwarnings("ignore", category=UserWarning, module="pm4py")
warnings.filterwarnings("ignore", category=FutureWarning, module="pm4py")

# print("🔇 Warnings suppressed for cleaner output")

#### Load and Validate the Synthetic BPMN Model

- Set `bpmn_path` to the fixed synthetic BPMN model used for the controlled activity-mapping validation.
- Validate the BPMN XML against the BPMN 2.0 XSD before converting it to a Petri net.

In [3]:
def resolve_existing_path(*relative_options):
    for relative_path in relative_options:
        path = Path(relative_path)
        if path.exists():
            return path
    raise FileNotFoundError(
        "Could not find any of these paths: " + ", ".join(str(path) for path in relative_options)
    )

bpmn_path = str(ACTIVITY_MAPPING_DIR / "activity-mapping-model.bpmn")
xsd_file_path = str(resolve_existing_path(
    "BPMN20/BPMN20.xsd",
    "../BPMN20/BPMN20.xsd",
    "../../BPMN20/BPMN20.xsd",
))

with open(bpmn_path, 'rb') as f:
    bpmn_xml = etree.parse(f)

with open(xsd_file_path, 'rb') as f:
    schema_root = etree.parse(f)
    schema = etree.XMLSchema(schema_root)

is_valid = schema.validate(bpmn_xml)
print("BPMN syntax valid?", is_valid)
print(f"Reference BPMN Model: {bpmn_path}")

if not is_valid:
    print("\nValidation Errors:")
    validation_error_messages = []
    for error in schema.error_log:
        error_message = f"Line {error.line}: {error.message}"
        validation_error_messages.append(error_message)
        print(error_message)
    raise RuntimeError(
        "BPMN syntax is invalid. " + " | ".join(validation_error_messages)
    )


BPMN syntax valid? True
Reference BPMN Model: D:\Mestrado\Tese\PM4PY\platform\backend\evaluation\activity-mapping\activity-mapping-model.bpmn


#### Import BPMN Model

Load the validated BPMN model with PM4Py. This model provides the reference activity labels and stable BPMN activity identifiers used in the candidate rankings.

In [4]:
try:
    bpmn_model = pm4py.read_bpmn(bpmn_path)
except Exception as e:
    raise RuntimeError(f"Failed to load BPMN model from: {bpmn_path}") from e

#### Extract BPMN Activity Metadata

Extract BPMN activity IDs, labels, and cycle-time annotations where available. For this evaluation, stable BPMN IDs are important because duplicate or closely related labels must not be collapsed during ranking.

In [5]:
# Extract cycle time annotations from BPMN activities.
# This BPMN stores CT values in textAnnotation nodes linked to tasks by association.
cycle_time_pattern = re.compile(
    r"\bCT\s*=\s*([0-9]+(?:[\.,][0-9]+)?)\s*(?:working\s*)?days?\b",
    re.IGNORECASE,
)

namespaces = {
    "bpmn": "http://www.omg.org/spec/BPMN/20100524/MODEL"
}

def normalize_activity_key(label):
    if not isinstance(label, str):
        return ""
    label = label.strip().lower()
    label = label.replace('_', ' ').replace('-', ' ')
    label = re.sub(r'[^\w\u00C0-\u017F&]+', ' ', label)
    label = re.sub(r'\s+', ' ', label)
    return label.strip()

def parse_cycle_time_days(value):
    days = float(value.replace(",", "."))
    return int(days) if days.is_integer() else days

activity_xpath = " | ".join([
    "//bpmn:task",
    "//bpmn:userTask",
    "//bpmn:serviceTask",
    "//bpmn:sendTask",
    "//bpmn:receiveTask",
    "//bpmn:manualTask",
    "//bpmn:businessRuleTask",
    "//bpmn:scriptTask",
    "//bpmn:callActivity",
    "//bpmn:subProcess",
])

bpmn_tree = etree.parse(bpmn_path)
activity_cycle_times = []

activities_by_id = {
    activity.get("id"): activity
    for activity in bpmn_tree.xpath(activity_xpath, namespaces=namespaces)
}

cycle_times_by_annotation_id = {}
for annotation in bpmn_tree.xpath("//bpmn:textAnnotation", namespaces=namespaces):
    annotation_text = " ".join(annotation.xpath(".//bpmn:text/text()", namespaces=namespaces))
    match = cycle_time_pattern.search(annotation_text)
    if match:
        cycle_times_by_annotation_id[annotation.get("id")] = parse_cycle_time_days(match.group(1))

activity_ids_to_cycle_times = {}
for association in bpmn_tree.xpath("//bpmn:association", namespaces=namespaces):
    source_ref = association.get("sourceRef")
    target_ref = association.get("targetRef")

    if source_ref in activities_by_id and target_ref in cycle_times_by_annotation_id:
        activity_ids_to_cycle_times[source_ref] = cycle_times_by_annotation_id[target_ref]
    elif target_ref in activities_by_id and source_ref in cycle_times_by_annotation_id:
        activity_ids_to_cycle_times[target_ref] = cycle_times_by_annotation_id[source_ref]

for activity_id, cycle_time_days in activity_ids_to_cycle_times.items():
    activity = activities_by_id[activity_id]
    activity_name = activity.get("name") or activity_id
    clean_activity_name = re.sub(
        r"\s*\(?\bCT\s*=\s*[0-9]+(?:[\.,][0-9]+)?\s*(?:working\s*)?days?\)?",
        "",
        activity_name,
        flags=re.IGNORECASE,
    ).strip() or activity_name

    activity_cycle_times.append({
        "activity": clean_activity_name,
        "normalized_activity": normalize_activity_key(clean_activity_name),
        "cycle_time_days": cycle_time_days,
    })

if activity_cycle_times:
    cycle_times_by_activity = {
        item["activity"]: item["cycle_time_days"]
        for item in activity_cycle_times
    }

    cycle_times_activities = {
        item["normalized_activity"]: item["cycle_time_days"]
        for item in activity_cycle_times
    }

    print("✅ Extracted Cycle Times from BPMN Activities")
else:
    cycle_times_by_activity = {}
    cycle_times_activities = {}
    print("ℹ️ No Cycle Times to Extract from BPMN Activities")

ℹ️ No Cycle Times to Extract from BPMN Activities


#### Convert BPMN Model to Petri Net

Convert the BPMN model to a Petri net so the evaluation uses the same backend representation as the normal pipeline.

In [6]:
net, im, fm = pm4py.convert_to_petri_net(bpmn_model)
# pm4py.view_petri_net(net, im, fm)

#### Optional Petri Net Visualization

This section is optional for the controlled mapping evaluation. It can be used to inspect the converted Petri net, but it is not required for calculating activity-mapping rankings or validation metrics.

In [7]:
# Define directory and base filename
# output_dir = "../img"
# base_filename = "petri-net-v1.png"
# output_path = os.path.join(output_dir, base_filename)

# # Check if the file already exists and generate a unique filename
# counter = 1
# while os.path.exists(output_path):
#     counter += 1
#     output_path = os.path.join(output_dir, f"petri-net-v{counter}.png")

# # Save the Petri net visualization
# pm4py.save_vis_petri_net(net, im, fm, output_path)
# print(f"Petri net visualization saved to '{output_path}'")

#### Validate Petri Net Soundness

Check that the converted Petri net is sound before evaluating label mappings. If the model conversion is not sound, mapping results should not be interpreted as a valid controlled run.

In [8]:
from pm4py.algo.analysis.woflan import algorithm as woflan
import io
from contextlib import redirect_stdout, redirect_stderr

buf = io.StringIO()

with redirect_stdout(buf), redirect_stderr(buf):
    woflan_results = woflan.apply(net, im, fm)

# Extract soundness result
if isinstance(woflan_results, bool):
    is_sound = woflan_results
else:
    is_sound = woflan_results.get("is_sound", False)

if is_sound:
    print("✅ Petri Net sound")
else:
    print("❌ Petri Net NOT sound")
    raise RuntimeError("BPMN model is not sound. Please upload a sound BPMN model.")

✅ Petri Net sound


### Import Synthetic Event Log Variant

Load one synthetic event-log version for the controlled validation run. Across versions, only selected activity labels should change; process behaviour, traces, cases, lifecycle transitions, and timestamps should remain fixed so mapping differences can be attributed to terminology variation.

In [9]:
from pm4py.objects.log.importer.xes import importer as xes_importer
from pm4py.objects.conversion.log import converter as log_converter
from pm4py.objects.log.exporter.xes import exporter as xes_exporter
from contextlib import contextmanager, redirect_stdout, redirect_stderr

@contextmanager
def silence_tqdm_and_display():
    try:
        import tqdm
        from IPython import display as ipdisplay
    except Exception:
        yield
        return

    _tqdm_orig = getattr(tqdm, "tqdm", None)
    _tqdm_nb_orig = getattr(getattr(tqdm, "notebook", None), "tqdm", None) if hasattr(tqdm, "notebook") else None
    _ipydisplay_orig = getattr(ipdisplay, "display", None)

    def _silent_tqdm(*a, **k):
        k["disable"] = True
        if _tqdm_orig:
            return _tqdm_orig(*a, **k)
        return a[0] if a else iter([])

    try:
        if _tqdm_orig:
            tqdm.tqdm = _silent_tqdm
        if hasattr(tqdm, "notebook") and _tqdm_nb_orig:
            tqdm.notebook.tqdm = _silent_tqdm
        ipdisplay.display = lambda *a, **k: None
        yield
    finally:
        try:
            if _tqdm_orig:
                tqdm.tqdm = _tqdm_orig
            if _tqdm_nb_orig and hasattr(tqdm, "notebook"):
                tqdm.notebook.tqdm = _tqdm_nb_orig
        except Exception:
            pass
        if _ipydisplay_orig:
            ipdisplay.display = _ipydisplay_orig


def load_event_log(file_path):
    file_path = Path(file_path)
    if not file_path.exists():
        raise FileNotFoundError(f"Event log not found: {file_path}")

    file_extension = file_path.suffix.lower()
    buf_align = io.StringIO()

    if file_extension == ".xes":
        with silence_tqdm_and_display(), redirect_stdout(buf_align), redirect_stderr(buf_align):
            return xes_importer.apply(str(file_path), parameters={"show_progress_bar": False})

    if file_extension == ".csv":
        df = pandas.read_csv(file_path)
        if len(df.columns) == 1:
            df = pandas.read_csv(file_path, sep=";")

        required_columns = {"case_id", "atividade"}
        missing_required = required_columns - set(df.columns)
        if missing_required:
            raise ValueError(
                f"Missing required columns: {missing_required}. Available columns: {list(df.columns)}"
            )

        start_columns = [c for c in df.columns if "inicio" in c.lower() or "start" in c.lower()]
        end_columns = [c for c in df.columns if "final" in c.lower() or "end" in c.lower()]
        resource_columns = [c for c in df.columns if "recurso" in c.lower() or "papel" in c.lower()]

        if not start_columns:
            raise ValueError("No start timestamp column detected")

        events_list = []
        for _, row in df.iterrows():
            case_id = row["case_id"]
            activity = row["atividade"]
            start_time = pandas.to_datetime(row[start_columns[0]], errors="coerce")
            start_event = {
                "case:concept:name": case_id,
                "concept:name": activity,
                "time:timestamp": start_time,
                "lifecycle:transition": "start",
            }
            for res_col in resource_columns:
                start_event["org:resource"] = row[res_col]
            events_list.append(start_event)

            if end_columns:
                end_time = pandas.to_datetime(row[end_columns[0]], errors="coerce")
                end_event = {
                    "case:concept:name": case_id,
                    "concept:name": activity,
                    "time:timestamp": end_time,
                    "lifecycle:transition": "complete",
                }
                for res_col in resource_columns:
                    end_event["org:resource"] = row[res_col]
                events_list.append(end_event)

        return log_converter.apply(pandas.DataFrame(events_list))

    raise ValueError(f"Unsupported file type: {file_extension}. Please provide a .csv or .xes file.")


### Normalize and Compare Activity Labels

Normalize activity labels in the event log and Petri net using the same backend preprocessing rule. Mismatched log labels are then identified as the evaluation targets for semantic activity mapping.

In [10]:
def normalize_label(label):
    """
    Normalize labels using the same backend preprocessing rule.
    """
    if not isinstance(label, str):
        return ""
    label = label.strip().lower()
    label = label.replace('_', ' ').replace('-', ' ')
    label = re.sub(r'[^\w\u00C0-\u017F&]+', ' ', label)
    label = re.sub(r'\s+', ' ', label)
    return label.strip()


def get_log_activities(event_log):
    return {
        event["concept:name"]
        for trace in event_log
        for event in trace
        if "concept:name" in event
    }


def build_bpmn_activity_candidates():
    candidates = []
    for activity_id in sorted(activities_by_id):
        activity = activities_by_id[activity_id]
        original_label = activity.get("name") or activity_id
        normalized_label = normalize_label(original_label)
        if normalized_label:
            candidates.append({
                "bpmn_id": activity_id,
                "original_label": original_label,
                "normalized_label": normalized_label,
            })
    return candidates


bpmn_activity_candidates = build_bpmn_activity_candidates()
net_activities = {candidate["normalized_label"] for candidate in bpmn_activity_candidates}
print(f"Prepared {len(bpmn_activity_candidates)} BPMN activity candidates for validation.")


Prepared 12 BPMN activity candidates for validation.


### Generate Full Activity-Mapping Rankings and Metrics

For each mismatched log activity, this section scores every BPMN candidate activity and stores the complete ranked list in `activity_mapping_rankings`. The existing top-1 dictionaries, `mapping_suggestions` and `non_suggested_mappings`, are still produced for compatibility.

If `expected_mapping_path` points to an expected-outcome JSON file, the notebook also computes `activity_mapping_evaluation`, including top-1 agreement, top-3 agreement, suggestion coverage, threshold-decision agreement, incorrect recommendations above threshold, valid correspondences below threshold, and `No Label` handling. A template is provided in `example-expected-activity-mapping.json`.

In [11]:
print("Starting activity-mapping validation for all versions...", flush=True)
# Single-cell execution across all validation versions.
from difflib import SequenceMatcher
from sentence_transformers import SentenceTransformer, util
from rapidfuzz import fuzz
import spacy
import sys
import platform
import hashlib
import subprocess
import importlib.metadata

VERSION_IDS = ["v1", "v2", "v3", "v4"]
RESULTS_DIR = ACTIVITY_MAPPING_DIR / "results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print(f"Results directory: {RESULTS_DIR}", flush=True)
print("Loading semantic models...", flush=True)
model = SentenceTransformer(
    EMBEDDING_MODEL,
    cache_folder="./hf_cache",
    use_auth_token=None,
)
nlp = spacy.load(LINGUISTIC_PIPELINE)
print("Models loaded.", flush=True)


def normalize_for_similarity(label):
    label = (label or "").lower().strip()
    label = label.replace("_", " ").replace("-", " ")
    label = re.sub(r"[^a-z0-9áéíóúàãõçâêôäëïöüñ ]", "", label)
    label = re.sub(r"\s+", " ", label)
    return label


def extract_main_verb(label):
    normalized = normalize_for_similarity(label)
    return normalized.split()[0] if normalized.split() else ""


def extract_main_object(label):
    tokens = normalize_for_similarity(label).split()[1:]
    return " ".join(tokens)


def extract_verbs(label):
    doc = nlp(normalize_for_similarity(label))
    return [token.lemma_ for token in doc if token.pos_ == "VERB"]


SEMANTIC_VERB_PENALTY_CONFIG = {
    "high_similarity_floor": 0.75,
    "medium_similarity_floor": 0.55,
    "low_similarity_floor": 0.35,
    "high_similarity_penalty": 0.0,
    "medium_similarity_penalty": 0.05,
    "low_similarity_penalty": 0.15,
    "very_low_similarity_penalty": 0.30,
    "fallback_penalty": 0.0,
}


def calculate_semantic_verb_penalty(log_label, net_label, action_score, penalty_config=SEMANTIC_VERB_PENALTY_CONFIG):
    try:
        verbs_log = extract_verbs(log_label)
        verbs_net = extract_verbs(net_label)
    except Exception as exc:
        return penalty_config["fallback_penalty"], f"linguistic-fallback:{type(exc).__name__}"

    if not verbs_log or not verbs_net:
        return 0.0, "no-verbs"
    if verbs_log == verbs_net:
        return 0.0, "same-verb-lemmas"
    if action_score >= penalty_config["high_similarity_floor"]:
        return penalty_config["high_similarity_penalty"], "high-action-similarity"
    if action_score >= penalty_config["medium_similarity_floor"]:
        return penalty_config["medium_similarity_penalty"], "medium-action-similarity"
    if action_score >= penalty_config["low_similarity_floor"]:
        return penalty_config["low_similarity_penalty"], "low-action-similarity"
    return penalty_config["very_low_similarity_penalty"], "very-low-action-similarity"


def get_withholding_reason(confidence_ok, margin_ok, margin_applicable):
    if confidence_ok and margin_ok:
        return None
    reasons = []
    if not confidence_ok:
        reasons.append("confidence-below-theta")
    if margin_applicable and not margin_ok:
        reasons.append("margin-below-epsilon")
    return "+".join(reasons)


def ranking_key(candidate):
    return (
        -candidate["ranking_score"],
        -candidate["embedding_score"],
        -candidate["action_score"],
        -candidate["object_score"],
        -candidate["token_score"],
        -candidate["lexical_score"],
        candidate["bpmn_id"],
    )


def load_expected_outcomes(expected_mapping_path):
    expected_path = Path(expected_mapping_path)
    if not expected_path.exists():
        raise FileNotFoundError(f"Expected mapping JSON not found: {expected_path}")

    with expected_path.open("r", encoding="utf-8") as f:
        expected_payload = json.load(f)

    if "expected_mismatches" not in expected_payload:
        raise ValueError(
            "Expected mapping JSON must contain an 'expected_mismatches' object "
            "where each key is a mismatched log label and each value is the expected BPMN ID or null."
        )

    return expected_payload


def discover_version_files(version_id):
    version_dir = ACTIVITY_MAPPING_DIR / version_id
    if not version_dir.exists():
        raise FileNotFoundError(f"Version folder not found: {version_dir}")

    log_files = sorted(version_dir.glob("*.xes"))
    expected_files = sorted(version_dir.glob("*.json"))

    if len(log_files) != 1:
        raise RuntimeError(f"Expected exactly one .xes file in {version_dir}, found {len(log_files)}")
    if len(expected_files) != 1:
        raise RuntimeError(f"Expected exactly one .json expected mapping file in {version_dir}, found {len(expected_files)}")

    return {
        "version": version_id,
        "version_dir": version_dir,
        "log_path": log_files[0],
        "expected_mapping_path": expected_files[0],
    }


def score_activity_mappings(log_labels_list, candidate_activities):
    if not log_labels_list:
        return {}, {}, {}, []

    net_labels_list = [candidate["normalized_label"] for candidate in candidate_activities]

    log_embeddings = model.encode(log_labels_list, convert_to_tensor=True, show_progress_bar=False)
    net_embeddings = model.encode(net_labels_list, convert_to_tensor=True, show_progress_bar=False)
    full_label_sim = util.cos_sim(log_embeddings, net_embeddings)

    log_verbs = [extract_main_verb(lbl) for lbl in log_labels_list]
    net_verbs = [extract_main_verb(lbl) for lbl in net_labels_list]
    verb_emb_log = model.encode(log_verbs, convert_to_tensor=True, show_progress_bar=False)
    verb_emb_net = model.encode(net_verbs, convert_to_tensor=True, show_progress_bar=False)
    verb_sim = util.cos_sim(verb_emb_log, verb_emb_net)

    log_objects = [extract_main_object(lbl) for lbl in log_labels_list]
    net_objects = [extract_main_object(lbl) for lbl in net_labels_list]
    obj_emb_log = model.encode(log_objects, convert_to_tensor=True, show_progress_bar=False)
    obj_emb_net = model.encode(net_objects, convert_to_tensor=True, show_progress_bar=False)
    obj_sim = util.cos_sim(obj_emb_log, obj_emb_net)

    mapping_suggestions = {}
    non_suggested_mappings = {}
    activity_mapping_rankings = {}
    candidate_rows = []

    total_labels = len(log_labels_list)
    total_candidates = len(candidate_activities)
    print(
        f"   Scoring {total_labels} mismatched log activities against {total_candidates} BPMN candidates...",
        flush=True,
    )

    for i, log_label in enumerate(log_labels_list):
        print(f"      [{i + 1}/{total_labels}] {log_label}", flush=True)
        norm_log = normalize_for_similarity(log_label)
        candidate_scores = []

        for j, candidate_activity in enumerate(candidate_activities):
            net_label = candidate_activity["normalized_label"]
            norm_net = normalize_for_similarity(net_label)
            action_score = verb_sim[i][j].item()
            object_score = obj_sim[i][j].item()
            lexical_score = SequenceMatcher(None, norm_log, norm_net).ratio()
            token_score = fuzz.token_set_ratio(norm_log, norm_net) / 100.0
            embedding_score = full_label_sim[i][j].item()

            ranking_score = (
                ACTION_WEIGHT * action_score +
                OBJECT_WEIGHT * object_score +
                FULL_LABEL_WEIGHT * embedding_score +
                LEXICAL_WEIGHT * lexical_score +
                TOKEN_WEIGHT * token_score
            )

            verb_penalty, verb_penalty_reason = calculate_semantic_verb_penalty(log_label, net_label, action_score)
            confidence_score = ranking_score - verb_penalty

            candidate_scores.append({
                "bpmn_id": candidate_activity["bpmn_id"],
                "original_label": candidate_activity["original_label"],
                "normalized_label": net_label,
                "ranking_score": ranking_score,
                "confidence_score": confidence_score,
                "hybrid_score": confidence_score,
                "unadjusted_hybrid_score": ranking_score,
                "embedding_score": embedding_score,
                "action_score": action_score,
                "object_score": object_score,
                "lexical_score": lexical_score,
                "token_score": token_score,
                "verb_penalty": verb_penalty,
                "verb_penalty_reason": verb_penalty_reason,
            })

        ranked_candidates = sorted(candidate_scores, key=ranking_key)
        top_margin = None
        margin_applicable = len(ranked_candidates) > 1
        if margin_applicable:
            top_margin = ranked_candidates[0]["ranking_score"] - ranked_candidates[1]["ranking_score"]

        for rank_position, candidate in enumerate(ranked_candidates, start=1):
            confidence_ok = candidate["confidence_score"] >= CANDIDATE_THRESHOLD
            margin_ok = True if rank_position != 1 or not margin_applicable else top_margin >= MINIMUM_CANDIDATE_MARGIN
            above_threshold = confidence_ok and margin_ok if rank_position == 1 else confidence_ok
            candidate.update({
                "rank": rank_position,
                "candidate_margin": top_margin if rank_position == 1 else None,
                "margin_applicable": margin_applicable if rank_position == 1 else None,
                "threshold_condition": confidence_ok,
                "margin_condition": margin_ok if rank_position == 1 else None,
                "above_threshold": above_threshold,
                "withholding_reason": get_withholding_reason(confidence_ok, margin_ok, margin_applicable) if rank_position == 1 else None,
            })
            candidate_rows.append({
                "source_activity": log_label,
                **candidate,
            })

        activity_mapping_rankings[log_label] = {
            "log_label": log_label,
            "candidates": ranked_candidates,
        }

        best_candidate = ranked_candidates[0] if ranked_candidates else None
        if best_candidate:
            best_scores = (
                best_candidate["action_score"],
                best_candidate["object_score"],
                best_candidate["lexical_score"],
                best_candidate["token_score"],
            )
            if best_candidate["above_threshold"]:
                mapping_suggestions[log_label] = (
                    best_candidate["normalized_label"],
                    best_candidate["confidence_score"],
                    *best_scores,
                )
            else:
                non_suggested_mappings[log_label] = (
                    best_candidate["normalized_label"],
                    best_candidate["confidence_score"],
                    *best_scores,
                )
        else:
            non_suggested_mappings[log_label] = (None, 0, 0, 0, 0, 0)

    return mapping_suggestions, non_suggested_mappings, activity_mapping_rankings, candidate_rows


def evaluate_activity_mapping_rankings(activity_mapping_rankings, expected_payload):
    expected_mismatches = expected_payload.get("expected_mismatches", {})
    missing_label_placeholder = expected_payload.get("missing_label_placeholder")
    bpmn_labels_by_id = {candidate["bpmn_id"]: candidate["original_label"] for candidate in bpmn_activity_candidates}
    direct_candidates_by_label = {}
    for candidate in bpmn_activity_candidates:
        direct_candidates_by_label.setdefault(candidate["normalized_label"], []).append(candidate)

    evaluated = []
    direct_matches = []
    missing_label_placeholders = []
    outcome_rows = {
        "correct_high_confidence": [],
        "correct_lower_confidence": [],
        "incorrect_high_confidence": [],
        "incorrect_lower_confidence": [],
        "unrelated_lower_confidence": [],
        "unrelated_high_confidence": [],
    }

    for expected_log_label, expected_bpmn_id in expected_mismatches.items():
        normalized_log_label = normalize_label(expected_log_label)
        if missing_label_placeholder and normalized_log_label == normalize_label(missing_label_placeholder):
            row = {
                "source_activity": expected_log_label,
                "normalized_source_activity": normalized_log_label,
                "expected_bpmn_id": expected_bpmn_id,
                "expected_activity": None,
                "has_valid_correspondence": False,
                "suggested_bpmn_id": None,
                "suggested_activity": None,
                "suggested_normalized_activity": None,
                "hybrid_score": None,
                "unadjusted_hybrid_score": None,
                "action_score": None,
                "object_score": None,
                "lexical_score": None,
                "token_score": None,
                "ranking_score": None,
                "confidence_score": None,
                "embedding_score": None,
                "verb_penalty": None,
                "verb_penalty_reason": None,
                "candidate_margin": None,
                "margin_applicable": None,
                "threshold_condition": None,
                "margin_condition": None,
                "withholding_reason": None,
                "above_threshold": None,
                "top_1_match": None,
                "top_3_match": None,
                "threshold_decision_match": None,
                "ranking_available": False,
                "match_type": "missing",
                "outcome": "missing_label_placeholder",
                "expected_target_rank": None,
                "top_candidates": [],
            }
            evaluated.append(row)
            missing_label_placeholders.append(row)
            continue

        direct_candidates = direct_candidates_by_label.get(normalized_log_label, [])
        direct_candidate = next(
            (candidate for candidate in direct_candidates if candidate["bpmn_id"] == expected_bpmn_id),
            direct_candidates[0] if direct_candidates else None,
        )
        is_direct_match = bool(
            expected_bpmn_id is not None
            and direct_candidate
            and direct_candidate["bpmn_id"] == expected_bpmn_id
        )

        if is_direct_match:
            row = {
                "source_activity": expected_log_label,
                "normalized_source_activity": normalized_log_label,
                "expected_bpmn_id": expected_bpmn_id,
                "expected_activity": bpmn_labels_by_id.get(expected_bpmn_id),
                "has_valid_correspondence": True,
                "suggested_bpmn_id": direct_candidate["bpmn_id"],
                "suggested_activity": direct_candidate["original_label"],
                "suggested_normalized_activity": direct_candidate["normalized_label"],
                "hybrid_score": None,
                "unadjusted_hybrid_score": None,
                "action_score": None,
                "object_score": None,
                "lexical_score": None,
                "token_score": None,
                "ranking_score": None,
                "confidence_score": None,
                "embedding_score": None,
                "verb_penalty": None,
                "verb_penalty_reason": None,
                "candidate_margin": None,
                "margin_applicable": None,
                "threshold_condition": None,
                "margin_condition": None,
                "withholding_reason": None,
                "above_threshold": None,
                "top_1_match": True,
                "top_3_match": True,
                "threshold_decision_match": None,
                "ranking_available": False,
                "match_type": "direct",
                "outcome": "direct_correct",
                "expected_target_rank": None,
                "top_candidates": [],
            }
            evaluated.append(row)
            direct_matches.append(row)
            continue

        ranking_data = activity_mapping_rankings.get(normalized_log_label)
        candidates = ranking_data["candidates"] if ranking_data else []
        top_candidate = candidates[0] if candidates else None
        top_3 = candidates[:3]
        top_3_ids = [candidate["bpmn_id"] for candidate in top_3]
        has_valid_correspondence = expected_bpmn_id is not None
        above_threshold = bool(top_candidate and top_candidate["above_threshold"])
        top_1_match = bool(top_candidate and expected_bpmn_id and top_candidate["bpmn_id"] == expected_bpmn_id)
        top_3_match = bool(expected_bpmn_id and expected_bpmn_id in top_3_ids)
        expected_target_rank = next(
            (candidate["rank"] for candidate in candidates if candidate["bpmn_id"] == expected_bpmn_id),
            None,
        )

        if has_valid_correspondence and top_1_match and above_threshold:
            outcome = "correct_high_confidence"
        elif has_valid_correspondence and top_1_match and not above_threshold:
            outcome = "correct_lower_confidence"
        elif has_valid_correspondence and not top_1_match and above_threshold:
            outcome = "incorrect_high_confidence"
        elif has_valid_correspondence and not top_1_match and not above_threshold:
            outcome = "incorrect_lower_confidence"
        elif not has_valid_correspondence and above_threshold:
            outcome = "unrelated_high_confidence"
        else:
            outcome = "unrelated_lower_confidence"

        if has_valid_correspondence:
            threshold_decision_match = above_threshold and top_1_match
        else:
            threshold_decision_match = not above_threshold

        row = {
            "source_activity": expected_log_label,
            "normalized_source_activity": normalized_log_label,
            "expected_bpmn_id": expected_bpmn_id,
            "expected_activity": bpmn_labels_by_id.get(expected_bpmn_id),
            "has_valid_correspondence": has_valid_correspondence,
            "suggested_bpmn_id": top_candidate["bpmn_id"] if top_candidate else None,
            "suggested_activity": top_candidate["original_label"] if top_candidate else None,
            "suggested_normalized_activity": top_candidate["normalized_label"] if top_candidate else None,
            "hybrid_score": top_candidate["hybrid_score"] if top_candidate else None,
            "unadjusted_hybrid_score": top_candidate["unadjusted_hybrid_score"] if top_candidate else None,
            "action_score": top_candidate["action_score"] if top_candidate else None,
            "object_score": top_candidate["object_score"] if top_candidate else None,
            "lexical_score": top_candidate["lexical_score"] if top_candidate else None,
            "token_score": top_candidate["token_score"] if top_candidate else None,
            "ranking_score": top_candidate["ranking_score"] if top_candidate else None,
            "confidence_score": top_candidate["confidence_score"] if top_candidate else None,
            "embedding_score": top_candidate["embedding_score"] if top_candidate else None,
            "verb_penalty": top_candidate["verb_penalty"] if top_candidate else None,
            "verb_penalty_reason": top_candidate["verb_penalty_reason"] if top_candidate else None,
            "candidate_margin": top_candidate["candidate_margin"] if top_candidate else None,
            "margin_applicable": top_candidate["margin_applicable"] if top_candidate else None,
            "threshold_condition": top_candidate["threshold_condition"] if top_candidate else None,
            "margin_condition": top_candidate["margin_condition"] if top_candidate else None,
            "withholding_reason": top_candidate["withholding_reason"] if top_candidate else None,
            "above_threshold": above_threshold,
            "top_1_match": top_1_match,
            "top_3_match": top_3_match,
            "threshold_decision_match": threshold_decision_match,
            "ranking_available": ranking_data is not None,
            "match_type": "candidate",
            "outcome": outcome,
            "expected_target_rank": expected_target_rank,
            "top_candidates": [
                {
                    "rank": candidate["rank"],
                    "bpmn_id": candidate["bpmn_id"],
                    "activity": candidate["original_label"],
                    "ranking_score_h0": candidate["ranking_score"],
                    "confidence_score_hc": candidate["confidence_score"],
                    "semantic_verb_penalty_pv": candidate["verb_penalty"],
                    "action_score": candidate["action_score"],
                    "object_score": candidate["object_score"],
                    "lexical_score": candidate["lexical_score"],
                    "token_score": candidate["token_score"],
                    "ranking_score": candidate["ranking_score"],
                    "confidence_score": candidate["confidence_score"],
                    "embedding_score": candidate["embedding_score"],
                    "verb_penalty": candidate["verb_penalty"],
                    "verb_penalty_reason": candidate["verb_penalty_reason"],
                    "candidate_margin": candidate["candidate_margin"],
                    "margin_applicable": candidate["margin_applicable"],
                    "threshold_condition": candidate["threshold_condition"],
                    "margin_condition": candidate["margin_condition"],
                    "withholding_reason": candidate["withholding_reason"],
                    "above_threshold": candidate["above_threshold"],
                }
                for candidate in top_3
            ],
        }
        evaluated.append(row)
        outcome_rows[outcome].append(row)

    candidate_rows = [row for row in evaluated if row["match_type"] == "candidate"]
    valid_candidate_rows = [row for row in candidate_rows if row["has_valid_correspondence"]]
    unrelated_candidate_rows = [row for row in candidate_rows if not row["has_valid_correspondence"]]
    threshold_recommendations = [row for row in candidate_rows if row["above_threshold"]]
    tp = len(outcome_rows["correct_high_confidence"])
    fp = len(outcome_rows["incorrect_high_confidence"]) + len(outcome_rows["unrelated_high_confidence"])
    fn = (
        len(outcome_rows["correct_lower_confidence"])
        + len(outcome_rows["incorrect_lower_confidence"])
        + len(outcome_rows["incorrect_high_confidence"])
    )
    tn = len(outcome_rows["unrelated_lower_confidence"])

    def ratio(numerator, denominator):
        return None if denominator == 0 else numerator / denominator

    precision = ratio(tp, tp + fp)
    recall = ratio(tp, tp + fn)
    f1 = None if precision is None or recall is None or precision + recall == 0 else 2 * precision * recall / (precision + recall)

    metrics = {
        "version": expected_payload.get("version"),
        "threshold": CANDIDATE_THRESHOLD,
        "minimum_candidate_margin": MINIMUM_CANDIDATE_MARGIN,
        "direct_match_count": len(direct_matches),
        "direct_correct_count": len(direct_matches),
        "direct_match_accuracy": ratio(len(direct_matches), len(direct_matches)),
        "candidate_label_count": len(candidate_rows),
        "candidate_valid_target_count": len(valid_candidate_rows),
        "candidate_unrelated_count": len(unrelated_candidate_rows),
        "missing_label_placeholder_count": len(missing_label_placeholders),
        "evaluated_label_count": len(evaluated),
        "threshold_evaluated_label_count": len(candidate_rows),
        "high_confidence_count": len(threshold_recommendations),
        "above_threshold_count": len(threshold_recommendations),
        "lower_confidence_count": len(candidate_rows) - len(threshold_recommendations),
        "below_threshold_count": len(candidate_rows) - len(threshold_recommendations),
        "correct_top_1_count": len(outcome_rows["correct_high_confidence"]) + len(outcome_rows["correct_lower_confidence"]),
        "correct_high_confidence_count": len(outcome_rows["correct_high_confidence"]),
        "correct_lower_confidence_count": len(outcome_rows["correct_lower_confidence"]),
        "incorrect_top_1_count": len(outcome_rows["incorrect_high_confidence"]) + len(outcome_rows["incorrect_lower_confidence"]),
        "incorrect_high_confidence_count": len(outcome_rows["incorrect_high_confidence"]),
        "incorrect_lower_confidence_count": len(outcome_rows["incorrect_lower_confidence"]),
        "unrelated_lower_confidence_count": len(outcome_rows["unrelated_lower_confidence"]),
        "unrelated_high_confidence_count": len(outcome_rows["unrelated_high_confidence"]),
        "false_high_confidence_count": fp,
        "expected_targets_absent_from_top_k_count": sum(
            1 for row in valid_candidate_rows if row["expected_target_rank"] is None or row["expected_target_rank"] > 3
        ),
        "top_1_agreement": ratio(
            len(outcome_rows["correct_high_confidence"]) + len(outcome_rows["correct_lower_confidence"]),
            len(valid_candidate_rows),
        ),
        "top_3_agreement": ratio(sum(row["top_3_match"] for row in valid_candidate_rows), len(valid_candidate_rows)),
        "suggestion_coverage": ratio(len(threshold_recommendations), len(candidate_rows)),
        "threshold_precision": precision,
        "threshold_recall": recall,
        "threshold_f1": f1,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "tp": tp,
        "fp": fp,
        "fn": fn,
        "tn": tn,
        "unrelated_rejection_rate": ratio(tn, len(unrelated_candidate_rows)),
        # Backward-compatible names used by existing display helpers.
        "expected_count": len(valid_candidate_rows),
        "suggested_count": len(threshold_recommendations),
        "correct_count": tp,
        "incorrect_count": len(outcome_rows["incorrect_high_confidence"]) + len(outcome_rows["incorrect_lower_confidence"]),
        "missing_expected_count": sum(
            1 for row in valid_candidate_rows if row["expected_target_rank"] is None or row["expected_target_rank"] > 3
        ),
        "accuracy": ratio(
            len(outcome_rows["correct_high_confidence"]) + len(outcome_rows["correct_lower_confidence"]),
            len(valid_candidate_rows),
        ),
        "threshold_decision_agreement": ratio(tp + tn, len(candidate_rows)),
    }

    return {
        "metrics": metrics,
        "mappings": evaluated,
        "direct_matches": direct_matches,
        "missing_label_placeholders": missing_label_placeholders,
        "correct_high_confidence_candidates": outcome_rows["correct_high_confidence"],
        "correct_lower_confidence_candidates": outcome_rows["correct_lower_confidence"],
        "incorrect_high_confidence_candidates": outcome_rows["incorrect_high_confidence"],
        "incorrect_lower_confidence_candidates": outcome_rows["incorrect_lower_confidence"],
        "unrelated_lower_confidence": outcome_rows["unrelated_lower_confidence"],
        "unrelated_high_confidence": outcome_rows["unrelated_high_confidence"],
        "incorrect_mappings": outcome_rows["incorrect_high_confidence"] + outcome_rows["incorrect_lower_confidence"],
        "missing_expected_mappings": [
            row for row in valid_candidate_rows if row["expected_target_rank"] is None or row["expected_target_rank"] > 3
        ],
        "valid_below_threshold": outcome_rows["correct_lower_confidence"],
        "false_high_confidence_mappings": outcome_rows["incorrect_high_confidence"] + outcome_rows["unrelated_high_confidence"],
        "no_label_results": missing_label_placeholders,
    }


def get_missing_label_stats(event_log, missing_label_placeholder):
    empty_events = []
    for trace in event_log:
        for event in trace:
            activity = event.get("concept:name")
            if activity is None or str(activity).strip() == "":
                empty_events.append(event)

    complete_events = [
        event for event in empty_events
        if str(event.get("lifecycle:transition", "")).lower() == "complete"
    ]
    if complete_events:
        occurrence_count = len(complete_events)
    else:
        occurrence_count = len(empty_events)

    return {
        "runtime_placeholder": "No Label" if missing_label_placeholder and empty_events else None,
        "normalized_value": missing_label_placeholder if missing_label_placeholder and empty_events else None,
        "distinct_placeholder_labels": 1 if missing_label_placeholder and empty_events else 0,
        "activity_occurrences": occurrence_count,
        "lifecycle_events": len(empty_events),
        "candidate_generation": "excluded",
        "expected_outcome": "preserved as unmapped" if missing_label_placeholder and empty_events else None,
    }


def evaluate_version(version_id):
    print(f"\n--- {version_id}: discovering files ---", flush=True)
    version_files = discover_version_files(version_id)
    print(f"   Log: {version_files['log_path']}", flush=True)
    print(f"   Expected mapping: {version_files['expected_mapping_path']}", flush=True)

    print(f"--- {version_id}: loading event log ---", flush=True)
    event_log = load_event_log(version_files["log_path"])
    expected_payload = load_expected_outcomes(version_files["expected_mapping_path"])
    missing_label_stats = get_missing_label_stats(
        event_log,
        expected_payload.get("missing_label_placeholder"),
    )

    print(f"--- {version_id}: detecting activity mismatches ---", flush=True)
    original_log_activities = get_log_activities(event_log)
    normalized_log_activities = {normalize_label(activity) for activity in original_log_activities if normalize_label(activity)}
    missing_in_net = sorted(normalized_log_activities - net_activities)
    missing_in_log = sorted(net_activities - normalized_log_activities)
    print(
        f"   Log activities: {len(original_log_activities)} | mismatched log activities: {len(missing_in_net)} | BPMN-only activities: {len(missing_in_log)}",
        flush=True,
    )

    print(f"--- {version_id}: computing candidate rankings ---", flush=True)
    mapping_suggestions, non_suggested_mappings, rankings, candidate_rows = score_activity_mappings(
        missing_in_net,
        bpmn_activity_candidates,
    )
    print(f"--- {version_id}: evaluating against expected mapping ---", flush=True)
    evaluation = evaluate_activity_mapping_rankings(rankings, expected_payload)

    return {
        "version": version_id,
        "log_path": str(version_files["log_path"]),
        "expected_mapping_path": str(version_files["expected_mapping_path"]),
        "log_activity_count": len(original_log_activities),
        "bpmn_activity_count": len(bpmn_activity_candidates),
        "mismatched_log_activity_count": len(missing_in_net),
        "missing_in_net": missing_in_net,
        "missing_in_log": missing_in_log,
        "missing_label_stats": missing_label_stats,
        "mapping_suggestions": mapping_suggestions,
        "non_suggested_mappings": non_suggested_mappings,
        "activity_mapping_rankings": rankings,
        "candidate_rows": candidate_rows,
        **evaluation,
    }


def round_metric(value):
    return None if value is None else round(value, 4)


def build_overall_metrics(version_results):
    totals = {
        "versions_count": len(version_results),
        "direct_match_count": 0,
        "direct_correct_count": 0,
        "candidate_label_count": 0,
        "candidate_valid_target_count": 0,
        "candidate_unrelated_count": 0,
        "missing_label_placeholder_count": 0,
        "missing_label_activity_occurrences": 0,
        "missing_label_lifecycle_events": 0,
        "high_confidence_count": 0,
        "above_threshold_count": 0,
        "correct_top_1_count": 0,
        "correct_high_confidence_count": 0,
        "correct_lower_confidence_count": 0,
        "incorrect_top_1_count": 0,
        "false_high_confidence_count": 0,
        "unrelated_lower_confidence_count": 0,
        "unrelated_high_confidence_count": 0,
        "expected_targets_absent_from_top_k_count": 0,
        "tp": 0,
        "fp": 0,
        "fn": 0,
        "tn": 0,
    }
    top_3_hits = 0
    direct_or_top_1_correct = 0
    valid_expected_correspondences = 0

    for result in version_results:
        metrics = result["metrics"]
        for key in totals:
            if key in metrics:
                totals[key] += metrics.get(key, 0) or 0
        stats = result.get("missing_label_stats", {})
        totals["missing_label_activity_occurrences"] += stats.get("activity_occurrences", 0) or 0
        totals["missing_label_lifecycle_events"] += stats.get("lifecycle_events", 0) or 0
        top_3_hits += sum(
            1 for row in result["mappings"]
            if row.get("match_type") == "candidate" and row.get("has_valid_correspondence") and row.get("top_3_match")
        )
        direct_or_top_1_correct += metrics.get("direct_correct_count", 0) or 0
        direct_or_top_1_correct += metrics.get("correct_top_1_count", 0) or 0
        valid_expected_correspondences += metrics.get("direct_match_count", 0) or 0
        valid_expected_correspondences += metrics.get("candidate_valid_target_count", 0) or 0

    def ratio(numerator, denominator):
        return None if denominator == 0 else numerator / denominator

    precision = ratio(totals["tp"], totals["tp"] + totals["fp"])
    recall = ratio(totals["tp"], totals["tp"] + totals["fn"])
    f1 = None if precision is None or recall is None or precision + recall == 0 else 2 * precision * recall / (precision + recall)

    totals.update({
        "direct_match_agreement": ratio(totals["direct_correct_count"], totals["direct_match_count"]),
        "candidate_top_1_agreement": ratio(totals["correct_top_1_count"], totals["candidate_valid_target_count"]),
        "candidate_top_3_agreement": ratio(top_3_hits, totals["candidate_valid_target_count"]),
        "suggestion_coverage": ratio(totals["high_confidence_count"], totals["candidate_label_count"]),
        "valid_target_suggestion_coverage": ratio(totals["tp"], totals["candidate_valid_target_count"]),
        "threshold_precision": precision,
        "threshold_recall": recall,
        "threshold_f1": f1,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "unrelated_rejection_rate": ratio(totals["tn"], totals["candidate_unrelated_count"]),
        "direct_or_top_1_correspondence_agreement": ratio(direct_or_top_1_correct, valid_expected_correspondences),
        "direct_or_top_1_correct_count": direct_or_top_1_correct,
        "valid_expected_correspondences": valid_expected_correspondences,
        # Backward-compatible display names.
        "expected_count": totals["candidate_valid_target_count"],
        "suggested_count": totals["high_confidence_count"],
        "correct_count": totals["tp"],
        "incorrect_count": totals["incorrect_top_1_count"],
        "missing_expected_count": totals["expected_targets_absent_from_top_k_count"],
        "accuracy": ratio(totals["correct_top_1_count"], totals["candidate_valid_target_count"]),
    })
    return totals


def markdown_escape(value):
    if value is None:
        return ""
    return str(value).replace("|", "\\|").replace("\n", " ")


def format_number(value):
    if value is None:
        return "N/A"
    if isinstance(value, float):
        return f"{value:.4f}"
    return str(value)


def markdown_table(headers, rows):
    lines = [
        "| " + " | ".join(headers) + " |",
        "| " + " | ".join("---" for _ in headers) + " |",
    ]
    for row in rows:
        lines.append("| " + " | ".join(markdown_escape(format_number(value)) for value in row) + " |")
    return "\n".join(lines)


def build_summary_rows(version_results):
    rows = []
    for result in version_results:
        metrics = result["metrics"]
        rows.append({
            "version": result["version"],
            "expected_count": metrics["expected_count"],
            "suggested_count": metrics["suggested_count"],
            "correct_count": metrics["correct_count"],
            "incorrect_count": metrics["incorrect_count"],
            "missing_expected_count": metrics["missing_expected_count"],
            "direct_match_count": metrics["direct_match_count"],
            "top_1_agreement": round_metric(metrics["top_1_agreement"]),
            "top_3_agreement": round_metric(metrics["top_3_agreement"]),
            "threshold_precision": round_metric(metrics["threshold_precision"]),
            "threshold_recall": round_metric(metrics["threshold_recall"]),
            "threshold_f1": round_metric(metrics["threshold_f1"]),
        })
    return rows


def print_compact_sections(version_results, overall_metrics):
    print("\n=== Overall Results ===")
    print(f"Versions: {overall_metrics['versions_count']}")
    print(
        "Candidate Top-1: {top1} | High-Confidence Precision: {precision} | High-Confidence Recall: {recall} | High-Confidence F1: {f1}".format(
            top1=format_number(round_metric(overall_metrics["candidate_top_1_agreement"])),
            precision=format_number(round_metric(overall_metrics["threshold_precision"])),
            recall=format_number(round_metric(overall_metrics["threshold_recall"])),
            f1=format_number(round_metric(overall_metrics["threshold_f1"])),
        )
    )

    print("\n=== Version Summary ===")
    summary_df = pandas.DataFrame(build_summary_rows(version_results))
    try:
        display(summary_df)
    except NameError:
        print(summary_df.to_string(index=False))

    for result in version_results:
        version = result["version"]
        for title, rows in (
            ("Direct Matches", result["direct_matches"]),
            ("Incorrect Mappings", result["incorrect_mappings"]),
            ("Missing Expected Mappings", result["missing_expected_mappings"]),
            ("Valid Lower-Confidence Mappings", result["valid_below_threshold"]),
        ):
            print(f"\n=== {version} {title} ===")
            df = pandas.DataFrame(rows)
            if df.empty:
                print("None")
            else:
                try:
                    display(df)
                except NameError:
                    print(df.to_string(index=False))

    return summary_df


def file_sha256(path):
    path = Path(path)
    if not path.exists() or not path.is_file():
        return None
    digest = hashlib.sha256()
    with path.open("rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def package_version(package_name):
    try:
        return importlib.metadata.version(package_name)
    except importlib.metadata.PackageNotFoundError:
        return None


def command_output(args):
    try:
        return subprocess.check_output(args, text=True, stderr=subprocess.DEVNULL).strip()
    except Exception:
        return None


def repository_root():
    root = command_output(["git", "rev-parse", "--show-toplevel"])
    return Path(root) if root else Path.cwd()


def repo_relative_path(path):
    if path is None:
        return None
    path = Path(path)
    try:
        return path.resolve().relative_to(repository_root().resolve()).as_posix()
    except Exception:
        return str(path)


def runtime_weight_discrepancies(weights):
    expected = {
        "action": 0.30,
        "object": 0.20,
        "full_label": 0.25,
        "lexical": 0.10,
        "token": 0.15,
    }
    discrepancies = []
    for key, expected_value in expected.items():
        runtime_value = weights.get(key)
        if runtime_value is None or abs(runtime_value - expected_value) > 1e-9:
            discrepancies.append(f"{key}: expected {expected_value:.4f}, runtime {format_number(round_metric(runtime_value))}")
    if abs(CANDIDATE_THRESHOLD - 0.80) > 1e-9:
        discrepancies.append(f"candidate_threshold: expected 0.8000, runtime {CANDIDATE_THRESHOLD:.4f}")
    if abs(MINIMUM_CANDIDATE_MARGIN - 0.05) > 1e-9:
        discrepancies.append(f"minimum_candidate_margin: expected 0.0500, runtime {MINIMUM_CANDIDATE_MARGIN:.4f}")
    return discrepancies


def semantic_penalty_description(config):
    return (
        f"0.00 when no usable verb is identified; 0.00 when extracted verb lemmas agree; "
        f"0.00 when action similarity >= {config['high_similarity_floor']:.2f}; "
        f"{config['medium_similarity_penalty']:.2f} when action similarity >= {config['medium_similarity_floor']:.2f} and < {config['high_similarity_floor']:.2f}; "
        f"{config['low_similarity_penalty']:.2f} when action similarity >= {config['low_similarity_floor']:.2f} and < {config['medium_similarity_floor']:.2f}; "
        f"{config['very_low_similarity_penalty']:.2f} when action similarity < {config['low_similarity_floor']:.2f}"
    )


def tie_breaker_winner(tied_candidates):
    return sorted(
        tied_candidates,
        key=lambda candidate: (
            -candidate["embedding_score"],
            -candidate["action_score"],
            -candidate["object_score"],
            -candidate["token_score"],
            -candidate["lexical_score"],
            candidate["bpmn_id"],
        ),
    )[0]


def tie_resolution_field(tied_candidates, winner):
    fields = [
        ("resolved_by_full_label", "embedding_score"),
        ("resolved_by_action", "action_score"),
        ("resolved_by_object", "object_score"),
        ("resolved_by_token", "token_score"),
        ("resolved_by_lexical", "lexical_score"),
    ]
    remaining = list(tied_candidates)
    for counter_key, score_key in fields:
        max_score = max(candidate[score_key] for candidate in remaining)
        remaining = [candidate for candidate in remaining if candidate[score_key] == max_score]
        if len(remaining) == 1 and remaining[0]["bpmn_id"] == winner["bpmn_id"]:
            return counter_key
    return "resolved_by_stable_bpmn_id"


def build_tie_resolution_stats(version_results):
    stats = {
        "exact_h0_ties_observed": 0,
        "resolved_by_full_label": 0,
        "resolved_by_action": 0,
        "resolved_by_object": 0,
        "resolved_by_token": 0,
        "resolved_by_lexical": 0,
        "resolved_by_stable_bpmn_id": 0,
        "near_ties_withheld_because_margin_below_minimum": 0,
    }
    for result in version_results:
        for ranking in result.get("activity_mapping_rankings", {}).values():
            candidates = ranking.get("candidates", [])
            groups = {}
            for candidate in candidates:
                groups.setdefault(candidate.get("ranking_score"), []).append(candidate)
            for tied_candidates in groups.values():
                if len(tied_candidates) < 2:
                    continue
                stats["exact_h0_ties_observed"] += 1
                winner = tie_breaker_winner(tied_candidates)
                stats[tie_resolution_field(tied_candidates, winner)] += 1
        for row in result.get("mappings", []):
            if (
                row.get("match_type") == "candidate"
                and row.get("withholding_reason")
                and "margin-below-epsilon" in row.get("withholding_reason")
            ):
                stats["near_ties_withheld_because_margin_below_minimum"] += 1
    return stats


def tie_resolution_statement(stats):
    if stats["exact_h0_ties_observed"] == 0:
        return (
            "No exact equality between candidate ranking scores occurred in V1-V4. "
            "The deterministic component-score and stable-identifier tie-breaking sequence was therefore implemented but not exercised by this fixture set."
        )
    return "Exact ranking-score ties were observed and resolved using the documented deterministic tie-breaking sequence."


def env_file_path_for_report():
    return VALIDATION_ENV_PATH if VALIDATION_ENV_PATH.exists() else None


def build_run_metadata(report):
    env_path = env_file_path_for_report()
    notebook_path = ACTIVITY_MAPPING_DIR / "activity-mapping-pipeline.ipynb"
    linguistic_pipeline_version = None
    try:
        linguistic_pipeline_version = package_version(LINGUISTIC_PIPELINE.replace("_", "-"))
    except Exception:
        linguistic_pipeline_version = None

    return {
        "run_id": report["run_id"],
        "created_at": report["created_at"],
        "git_commit": command_output(["git", "rev-parse", "HEAD"]),
        "notebook_path": repo_relative_path(notebook_path),
        "notebook_sha256": file_sha256(notebook_path),
        "env_path": repo_relative_path(env_path) if env_path else None,
        "env_sha256": file_sha256(env_path) if env_path else None,
        "operating_system": platform.platform(),
        "python_version": sys.version.replace("\n", " "),
        "sentence_transformers_version": package_version("sentence-transformers"),
        "embedding_model": EMBEDDING_MODEL,
        "embedding_model_revision": resolve_embedding_model_revision(),
        "spacy_version": package_version("spacy"),
        "linguistic_pipeline": LINGUISTIC_PIPELINE,
        "linguistic_pipeline_version": linguistic_pipeline_version,
        "rapidfuzz_version": package_version("rapidfuzz"),
        "execution_device": str(getattr(model, "device", "N/A")),
        "weights": report["config"]["weights"],
        "candidate_threshold": report["config"]["threshold"],
        "minimum_candidate_margin": report["config"]["minimum_candidate_margin"],
        "configuration_discrepancies": runtime_weight_discrepancies(report["config"]["weights"]),
        "penalties": semantic_penalty_description(report["config"]["semantic_verb_penalty_config"]),
    }


def markdown_table_or_sentence(headers, rows, empty_sentence):
    if not rows:
        return empty_sentence
    return markdown_table(headers, rows)


def format_ratio(value, numerator, denominator):
    if value is None:
        return "N/A"
    return f"{format_number(round_metric(value))} ({numerator}/{denominator})"


def format_missing_label_count(overall):
    return (
        f"{overall['missing_label_placeholder_count']} distinct placeholder; "
        f"{overall['missing_label_activity_occurrences']} occurrences; "
        f"{overall['missing_label_lifecycle_events']} lifecycle events"
    )


def directory_sha256(path):
    path = Path(path)
    if not path.exists() or not path.is_dir():
        return None
    digest = hashlib.sha256()
    for file_path in sorted(item for item in path.rglob("*") if item.is_file()):
        digest.update(str(file_path.relative_to(path)).replace("\\", "/").encode("utf-8"))
        with file_path.open("rb") as f:
            for chunk in iter(lambda: f.read(1024 * 1024), b""):
                digest.update(chunk)
    return digest.hexdigest()


def resolve_embedding_model_revision():
    try:
        cache_roots = [
            ACTIVITY_MAPPING_DIR / "hf_cache" / "hub",
            Path("hf_cache") / "hub",
            Path.home() / ".cache" / "huggingface" / "hub",
        ]
        model_root_name = f"models--{EMBEDDING_MODEL.replace('/', '--')}"
        for cache_root in cache_roots:
            model_root = cache_root / model_root_name
            snapshots_dir = model_root / "snapshots"
            if snapshots_dir.exists():
                snapshots = sorted(
                    [path for path in snapshots_dir.iterdir() if path.is_dir()],
                    key=lambda path: path.stat().st_mtime,
                    reverse=True,
                )
                if snapshots:
                    return snapshots[0].name
            model_hash = directory_sha256(model_root)
            if model_hash:
                return f"model-directory-sha256:{model_hash}"

        model_card_vars = getattr(model, "_model_card_vars", None)
        if isinstance(model_card_vars, dict) and model_card_vars.get("model_id"):
            return model_card_vars["model_id"]
        return "N/A"
    except Exception:
        return "N/A"


def compact_score(candidate):
    if not candidate:
        return "N/A"
    return (
        f"{candidate.get('activity')} (`{candidate.get('bpmn_id')}`), "
        f"h0={format_number(round_metric(candidate.get('ranking_score_h0', candidate.get('ranking_score'))))}, "
        f"hc={format_number(round_metric(candidate.get('confidence_score_hc', candidate.get('confidence_score'))))}, "
        f"pv={format_number(round_metric(candidate.get('semantic_verb_penalty_pv', candidate.get('verb_penalty'))))}"
    )


def direct_rows(rows):
    return [
        [
            row.get("source_activity"),
            row.get("normalized_source_activity"),
            row.get("suggested_activity"),
            row.get("suggested_bpmn_id"),
            "direct",
            row.get("top_1_match"),
            "N/A",
            "N/A",
        ]
        for row in rows
    ]


def candidate_rows(rows):
    return [
        [
            row.get("source_activity"),
            row.get("expected_activity") or "null",
            row.get("expected_bpmn_id") or "null",
            row.get("suggested_activity"),
            row.get("suggested_bpmn_id"),
            row.get("outcome"),
            row.get("expected_target_rank"),
            row.get("above_threshold"),
            round_metric(row.get("confidence_score")),
            round_metric(row.get("ranking_score")),
            round_metric(row.get("embedding_score")),
            round_metric(row.get("action_score")),
            round_metric(row.get("object_score")),
            round_metric(row.get("lexical_score")),
            round_metric(row.get("token_score")),
            round_metric(row.get("verb_penalty")),
            row.get("verb_penalty_reason"),
            round_metric(row.get("candidate_margin")),
            row.get("withholding_reason"),
        ]
        for row in rows
    ]


def missing_rows(result):
    stats = result.get("missing_label_stats", {})
    placeholders = result.get("missing_label_placeholders", [])
    if not placeholders and not stats.get("lifecycle_events"):
        return []
    return [[
        "empty source activity label",
        stats.get("runtime_placeholder"),
        stats.get("normalized_value"),
        stats.get("distinct_placeholder_labels"),
        stats.get("activity_occurrences"),
        stats.get("lifecycle_events"),
        stats.get("candidate_generation"),
        stats.get("expected_outcome"),
    ]]


def top_three_rows(rows):
    output = []
    for row in rows:
        top_candidates = row.get("top_candidates", [])
        output.append([
            row.get("source_activity"),
            row.get("expected_activity") or "null",
            row.get("expected_bpmn_id") or "null",
            row.get("expected_target_rank"),
            compact_score(top_candidates[0] if len(top_candidates) > 0 else None),
            compact_score(top_candidates[1] if len(top_candidates) > 1 else None),
            compact_score(top_candidates[2] if len(top_candidates) > 2 else None),
        ])
    return output


def version_outcome(metrics):
    if metrics["candidate_label_count"] == 0 and metrics["direct_match_count"] > 0:
        return "direct only"
    if metrics["false_high_confidence_count"] == 0 and metrics["candidate_unrelated_count"] > 0:
        return "unrelated kept lower confidence"
    if metrics["top_1_agreement"] == 1:
        return "candidate top-1 recovered"
    if metrics["top_3_agreement"] == 1:
        return "top-3 recovered"
    return "mixed"


def build_version_conclusion(result):
    version = result["version"]
    metrics = result["metrics"]
    if version == "v1":
        return "Exact label correspondences were resolved completely through direct matching without invoking similarity-assisted recommendation."
    if version == "v2":
        return (
            "The controlled formatting and lexical variants were recovered completely. Four formatting variants were resolved "
            "through direct normalised matching, while all eight remaining labels received the correct first-ranked candidate "
            "and satisfied the high-confidence criterion. No incorrect or false-high-confidence recommendation was produced."
        )
    if version == "v3":
        return (
            "The expected semantic correspondence appeared within the first three candidates for every evaluated label and occupied "
            "the first position for 10 of the 12 labels. None of the first-ranked candidates satisfied the fixed high-confidence "
            "criterion. The results therefore show strong candidate retrieval but limited high-confidence coverage for the evaluated "
            "semantic variations."
        )
    if version == "v4":
        return (
            "All unrelated labels remained below the configured high-confidence criterion and were therefore not presented as "
            "high-confidence recommendations. No false-high-confidence recommendation was produced, while empty source labels were "
            "preserved through the designated placeholder and excluded from candidate generation. Lower-confidence candidates were "
            "still generated and retained for human inspection."
        )
    return (
        f"Version {version} produced top-1 agreement {format_number(round_metric(metrics.get('top_1_agreement')))} "
        f"and high-confidence precision {format_number(round_metric(metrics.get('threshold_precision')))}."
    )


def build_overall_conclusion(overall):
    return (
        "The controlled evaluation showed complete direct matching, complete top-1 recovery for the evaluated lexical variants, "
        "and top-1 recovery for 10 of the 12 evaluated semantic variants. Across V2 and V3, candidate top-1 agreement was "
        f"{overall['correct_top_1_count']} of {overall['candidate_valid_target_count']}, while every valid-target candidate appeared "
        "within the first three ranked candidates. The fixed high-confidence criterion of "
        f"hc >= {format_number(CANDIDATE_THRESHOLD)} and candidate margin >= {format_number(MINIMUM_CANDIDATE_MARGIN)} produced "
        f"{overall['false_high_confidence_count']} false-high-confidence recommendations "
        f"and {format_number(round_metric(overall['valid_target_suggestion_coverage']))} valid-target high-confidence coverage. "
        "Lower-confidence candidates were still generated and retained for human inspection. These results characterise the evaluated "
        "English model, linguistic pipeline, fixed weights, semantic verb penalty, threshold, margin, and synthetic vocabulary; they "
        "do not establish equivalent performance for other configurations or datasets."
    )


def build_markdown_report(report):
    config = report["config"]
    bpmn = report["bpmn"]
    overall = report["overall_metrics"]
    version_results = report["versions"]
    metadata = report.get("metadata") or build_run_metadata(report)
    tie_stats = build_tie_resolution_stats(version_results)
    discrepancy_text = "; ".join(metadata["configuration_discrepancies"]) if metadata["configuration_discrepancies"] else "None"

    version_hash_rows = []
    for result in version_results:
        version_hash_rows.append([
            result["version"],
            result["log_path"],
            file_sha256(result["log_path"]),
            result["expected_mapping_path"],
            file_sha256(result["expected_mapping_path"]),
        ])

    lines = [
        "# Activity-Mapping Evaluation Report",
        "",
        "## 1. Run Metadata",
        "",
        markdown_table(
            ["Field", "Value"],
            [
                ["Run identifier", metadata["run_id"]],
                ["Timestamp", metadata["created_at"]],
                ["Prototype Git commit or release", metadata["git_commit"]],
                ["Evaluation notebook", metadata["notebook_path"]],
                ["Evaluation notebook SHA-256", metadata["notebook_sha256"]],
                ["Validation .env", metadata["env_path"]],
                ["Validation .env SHA-256", metadata["env_sha256"]],
                ["Operating system", metadata["operating_system"]],
                ["Python version", metadata["python_version"]],
                ["sentence-transformers version", metadata["sentence_transformers_version"]],
                ["Embedding model", metadata["embedding_model"]],
                ["Embedding model revision", metadata["embedding_model_revision"]],
                ["spaCy version", metadata["spacy_version"]],
                ["Linguistic pipeline", metadata["linguistic_pipeline"]],
                ["Linguistic pipeline version", metadata["linguistic_pipeline_version"]],
                ["RapidFuzz version", metadata["rapidfuzz_version"]],
                ["CPU/GPU execution device", metadata["execution_device"]],
                ["Action weight", metadata["weights"]["action"]],
                ["Object weight", metadata["weights"]["object"]],
                ["Full-label semantic weight", metadata["weights"]["full_label"]],
                ["Lexical weight", metadata["weights"]["lexical"]],
                ["Token weight", metadata["weights"]["token"]],
                ["Candidate threshold", metadata["candidate_threshold"]],
                ["Minimum candidate margin", metadata["minimum_candidate_margin"]],
                ["Semantic verb penalty", metadata["penalties"]],
                ["Configuration discrepancy flag", discrepancy_text],
            ],
        ),
        "",
        "### Input Hashes",
        "",
        markdown_table(
            ["Artifact", "Path", "SHA-256"],
            [["BPMN", bpmn["path"], file_sha256(bpmn["path"])]],
        ),
        "",
        markdown_table(
            ["Version", "XES path", "XES SHA-256", "Expected JSON path", "Expected JSON SHA-256"],
            version_hash_rows,
        ),
        "",
        "## 2. BPMN Reference Model",
        "",
        markdown_table(
            ["Field", "Value"],
            [
                ["Model filename", Path(bpmn["path"]).name],
                ["Model SHA-256", file_sha256(bpmn["path"])],
                ["Number of activities", bpmn["activities_count"]],
                ["Stable identifiers used", "Yes"],
            ],
        ),
        "",
        "## 3. Overall Summary",
        "",
        markdown_table(
            ["Measure", "Value"],
            [
                ["Direct-match agreement", format_ratio(overall["direct_match_agreement"], overall["direct_correct_count"], overall["direct_match_count"])],
                ["Candidate top-1 agreement", format_ratio(overall["candidate_top_1_agreement"], overall["correct_top_1_count"], overall["candidate_valid_target_count"])],
                ["Candidate top-3 agreement", format_ratio(overall["candidate_top_3_agreement"], overall["candidate_valid_target_count"], overall["candidate_valid_target_count"])],
                ["Overall high-confidence coverage", format_ratio(overall["suggestion_coverage"], overall["high_confidence_count"], overall["candidate_label_count"])],
                ["Valid-target high-confidence coverage", format_ratio(overall["valid_target_suggestion_coverage"], overall["tp"], overall["candidate_valid_target_count"])],
                ["High-confidence precision", format_ratio(overall["threshold_precision"], overall["tp"], overall["tp"] + overall["fp"])],
                ["High-confidence recall", format_ratio(overall["threshold_recall"], overall["tp"], overall["tp"] + overall["fn"])],
                ["High-confidence F1", round_metric(overall["threshold_f1"])],
                ["False-high-confidence count", overall["false_high_confidence_count"]],
                ["Unrelated-label high-confidence withholding rate", format_ratio(overall["unrelated_rejection_rate"], overall["tn"], overall["candidate_unrelated_count"])],
                ["Missing-label handling count", format_missing_label_count(overall)],
                ["Direct-or-top-1 correspondence agreement", format_ratio(overall["direct_or_top_1_correspondence_agreement"], overall["direct_or_top_1_correct_count"], overall["valid_expected_correspondences"])],
            ],
        ),
        "",
        (
            "Candidate-ranking agreement is calculated only for the "
            f"{overall['candidate_valid_target_count']} labels with predefined BPMN targets in V2 and V3. "
            "Overall high-confidence coverage is calculated across all "
            f"{overall['candidate_label_count']} non-empty labels entering candidate generation in V2-V4; therefore, "
            f"{overall['high_confidence_count']} of {overall['candidate_label_count']} labels satisfied the high-confidence criterion "
            f"({format_number(round_metric(overall['suggestion_coverage']))}). Restricted to labels with predefined BPMN targets, "
            f"coverage was {overall['tp']} of {overall['candidate_valid_target_count']} "
            f"({format_number(round_metric(overall['valid_target_suggestion_coverage']))}). The missing-label placeholder was excluded from both populations."
        ),
        "",
        "### High-Confidence Decision Confusion Matrix",
        "",
        markdown_table(
            ["TP", "FP", "FN", "TN", "High-confidence precision", "High-confidence recall", "High-confidence F1"],
            [[
                overall["tp"], overall["fp"], overall["fn"], overall["tn"],
                round_metric(overall["threshold_precision"]),
                round_metric(overall["threshold_recall"]),
                round_metric(overall["threshold_f1"]),
            ]],
        ),
        "",
        "### Tie-Resolution Coverage",
        "",
        markdown_table(
            ["Measure", "Count"],
            [
                ["Exact h0 ties observed", tie_stats["exact_h0_ties_observed"]],
                ["Resolved by sf", tie_stats["resolved_by_full_label"]],
                ["Resolved by sv", tie_stats["resolved_by_action"]],
                ["Resolved by so", tie_stats["resolved_by_object"]],
                ["Resolved by st", tie_stats["resolved_by_token"]],
                ["Resolved by sl", tie_stats["resolved_by_lexical"]],
                ["Resolved by stable BPMN identifier", tie_stats["resolved_by_stable_bpmn_id"]],
                ["Near-ties classified as lower confidence because Δ < ε", tie_stats["near_ties_withheld_because_margin_below_minimum"]],
            ],
        ),
        "",
        tie_resolution_statement(tie_stats),
        "",
        "## 4. Per-Version Summary",
        "",
        markdown_table(
            ["Version", "Direct", "Candidate labels", "Top-1", "Top-3", "High-confidence candidates", "False high-confidence", "Missing labels", "Outcome"],
            [
                [
                    result["version"],
                    f"{result['metrics']['direct_correct_count']}/{result['metrics']['direct_match_count']}" if result["metrics"]["direct_match_count"] else "N/A",
                    result["metrics"]["candidate_label_count"],
                    round_metric(result["metrics"]["top_1_agreement"]),
                    round_metric(result["metrics"]["top_3_agreement"]),
                    f"{result['metrics']['high_confidence_count']}/{result['metrics']['candidate_label_count']}" if result["metrics"]["candidate_label_count"] else "N/A",
                    result["metrics"]["false_high_confidence_count"],
                    result.get("missing_label_stats", {}).get("distinct_placeholder_labels", 0),
                    version_outcome(result["metrics"]),
                ]
                for result in version_results
            ],
        ),
        "",
        "## 5. Per-Version Results",
    ]

    candidate_headers = [
        "Source", "Expected activity", "Expected ID", "Top candidate", "Top ID", "Outcome", "Expected rank",
        "High confidence", "Confidence score hc", "Ranking score h0", "Full-label sf", "s_v", "s_o", "s_l", "s_t",
        "Verb penalty", "Verb penalty reason", "Candidate margin", "Withholding reason",
    ]

    for result in version_results:
        all_candidate_rows = [row for row in result["mappings"] if row.get("match_type") == "candidate"]
        lines.extend([
            "",
            f"### {result['version']}",
            "",
            markdown_table(
                ["Metric", "Value"],
                [
                    ["Direct matches", result["metrics"]["direct_match_count"]],
                    ["Candidate-generated labels", result["metrics"]["candidate_label_count"]],
                    ["Correct top-1 candidates", result["metrics"]["correct_top_1_count"]],
                    ["Top-1 agreement", round_metric(result["metrics"]["top_1_agreement"])],
                    ["Top-3 agreement", round_metric(result["metrics"]["top_3_agreement"])],
                    ["High-confidence candidates", result["metrics"]["high_confidence_count"]],
                    ["High-confidence coverage", round_metric(result["metrics"]["suggestion_coverage"])],
                    ["High-confidence precision", round_metric(result["metrics"]["threshold_precision"])],
                    ["High-confidence recall", round_metric(result["metrics"]["threshold_recall"])],
                    ["High-confidence F1", round_metric(result["metrics"]["threshold_f1"])],
                    ["TP", result["metrics"]["tp"]],
                    ["FP", result["metrics"]["fp"]],
                    ["FN", result["metrics"]["fn"]],
                    ["TN", result["metrics"]["tn"]],
                ],
            ),
            "",
            "#### 1. Direct Matches",
            "",
            markdown_table_or_sentence(
                ["Source", "Normalised source", "Matched BPMN activity", "Matched BPMN ID", "Mapping type", "Correct", "Ranking score h0", "High-confidence decision"],
                direct_rows(result["direct_matches"]),
                "No direct matches were observed.",
            ),
            "",
            "#### 2. Correct High-Confidence Candidates",
            "",
            markdown_table_or_sentence(candidate_headers, candidate_rows(result["correct_high_confidence_candidates"]), "No correct high-confidence candidates were observed."),
            "",
            "#### 3. Correct Lower-Confidence Candidates",
            "",
            markdown_table_or_sentence(candidate_headers, candidate_rows(result["correct_lower_confidence_candidates"]), "No correct lower-confidence candidates were observed."),
            "",
            "#### 4. Incorrect Top-1 Candidates",
            "",
            markdown_table_or_sentence(candidate_headers, candidate_rows(result["incorrect_mappings"]), "No incorrect top-1 candidates were observed."),
            "",
            "#### 5. Unrelated Labels Kept Lower Confidence",
            "",
            markdown_table_or_sentence(candidate_headers, candidate_rows(result["unrelated_lower_confidence"]), "No unrelated labels were kept lower confidence."),
            "",
            "#### 6. False-High-Confidence Unrelated or Incorrect Candidates",
            "",
            markdown_table_or_sentence(candidate_headers, candidate_rows(result["false_high_confidence_mappings"]), "No false-high-confidence mappings were observed."),
            "",
            "#### 7. Missing-Label Placeholders",
            "",
            markdown_table_or_sentence(
                ["Source label value", "Runtime display placeholder", "Internal placeholder key", "Distinct placeholders", "Activity occurrences", "Lifecycle events", "Candidate generation", "Expected outcome"],
                missing_rows(result),
                "No missing-label placeholders were observed.",
            ),
            "",
            "#### 8. Top-Three Candidate Details",
            "",
            markdown_table_or_sentence(
                ["Source", "Expected activity", "Expected ID", "Expected rank", "Top 1 by h0", "Top 2 by h0", "Top 3 by h0"],
                top_three_rows(all_candidate_rows),
                "No candidate-generated labels were evaluated.",
            ),
            "",
            "#### Version Conclusion",
            "",
            build_version_conclusion(result),
        ])

    lines.extend([
        "",
        "## 6. Evaluation Conclusion",
        "",
        build_overall_conclusion(overall),
    ])

    return "\n".join(lines) + "\n"


run_id = datetime.now().strftime("%Y-%m-%d-%H%M%S")
created_at = datetime.now().astimezone().isoformat(timespec="seconds")

version_results = []
for version_id in VERSION_IDS:
    print(f"\nRunning activity-mapping validation for {version_id}...", flush=True)
    version_result = evaluate_version(version_id)
    version_results.append(version_result)
    metrics = version_result["metrics"]
    print(
        f"{version_id}: top1={format_number(round_metric(metrics['top_1_agreement']))}, "
        f"top3={format_number(round_metric(metrics['top_3_agreement']))}, "
        f"high_confidence_precision={format_number(round_metric(metrics['threshold_precision']))}, "
        f"high_confidence_recall={format_number(round_metric(metrics['threshold_recall']))}, "
        f"high_confidence_f1={format_number(round_metric(metrics['threshold_f1']))}",
        flush=True,
    )

overall_metrics = build_overall_metrics(version_results)
tie_resolution_stats = build_tie_resolution_stats(version_results)
summary_df = print_compact_sections(version_results, overall_metrics)

activity_mapping_report = {
    "run_id": run_id,
    "created_at": created_at,
    "config": {
        "embedding_model": EMBEDDING_MODEL,
        "linguistic_pipeline": LINGUISTIC_PIPELINE,
        "weights": {
            "action": ACTION_WEIGHT,
            "object": OBJECT_WEIGHT,
            "full_label": FULL_LABEL_WEIGHT,
            "lexical": LEXICAL_WEIGHT,
            "token": TOKEN_WEIGHT,
        },
        "semantic_verb_penalty_config": SEMANTIC_VERB_PENALTY_CONFIG,
        "threshold": CANDIDATE_THRESHOLD,
        "minimum_candidate_margin": MINIMUM_CANDIDATE_MARGIN,
    },
    "bpmn": {
        "path": bpmn_path,
        "model_filename": Path(bpmn_path).name,
        "sha256": file_sha256(bpmn_path),
        "activities_count": len(bpmn_activity_candidates),
        "stable_identifiers_used": True,
    },
    "versions": version_results,
    "overall_metrics": overall_metrics,
    "tie_resolution_stats": tie_resolution_stats,
}

activity_mapping_report["metadata"] = build_run_metadata(activity_mapping_report)
activity_mapping_report["input_hashes"] = {
    "bpmn": {
        "path": bpmn_path,
        "sha256": file_sha256(bpmn_path),
    },
    "versions": [
        {
            "version": result["version"],
            "xes_path": result["log_path"],
            "xes_sha256": file_sha256(result["log_path"]),
            "expected_mapping_path": result["expected_mapping_path"],
            "expected_mapping_sha256": file_sha256(result["expected_mapping_path"]),
        }
        for result in version_results
    ],
}

run_output_dir = RESULTS_DIR / run_id
run_output_dir.mkdir(parents=True, exist_ok=True)

json_output_path = run_output_dir / f"activity-mapping-evaluation-{run_id}.json"
markdown_output_path = run_output_dir / f"activity-mapping-evaluation-{run_id}.md"

with json_output_path.open("w", encoding="utf-8") as f:
    json.dump(activity_mapping_report, f, ensure_ascii=False, indent=2)

markdown_output_path.write_text(build_markdown_report(activity_mapping_report), encoding="utf-8")

print("\n=== Persisted Output Files ===")
print(f"JSON: {json_output_path}", flush=True)
print(f"Markdown: {markdown_output_path}", flush=True)


Starting activity-mapping validation for all versions...
Results directory: D:\Mestrado\Tese\PM4PY\platform\backend\evaluation\activity-mapping\results
Loading semantic models...
Models loaded.

Running activity-mapping validation for v1...

--- v1: discovering files ---
   Log: D:\Mestrado\Tese\PM4PY\platform\backend\evaluation\activity-mapping\v1\v1-exact-labels.xes
   Expected mapping: D:\Mestrado\Tese\PM4PY\platform\backend\evaluation\activity-mapping\v1\v1-exact-labels.json
--- v1: loading event log ---
--- v1: detecting activity mismatches ---
   Log activities: 12 | mismatched log activities: 0 | BPMN-only activities: 0
--- v1: computing candidate rankings ---
--- v1: evaluating against expected mapping ---
v1: top1=N/A, top3=N/A, high_confidence_precision=N/A, high_confidence_recall=N/A, high_confidence_f1=N/A

Running activity-mapping validation for v2...

--- v2: discovering files ---
   Log: D:\Mestrado\Tese\PM4PY\platform\backend\evaluation\activity-mapping\v2\v2-lexical-va

,version,expected_count,suggested_count,correct_count,incorrect_count,missing_expected_count,direct_match_count,top_1_agreement,top_3_agreement,threshold_precision,threshold_recall,threshold_f1
0,v1,0,0,0,0,0,12,NaN,NaN,NaN,NaN,NaN
1,v2,8,8,8,0,0,4,1.0000,1.0,1.0,1.0,1.0
2,v3,12,0,0,2,0,0,0.8333,1.0,NaN,0.0,NaN
3,v4,0,0,0,0,0,0,NaN,NaN,NaN,NaN,NaN



=== v1 Direct Matches ===


,source_activity,normalized_source_activity,expected_bpmn_id,expected_activity,has_valid_correspondence,suggested_bpmn_id,suggested_activity,suggested_normalized_activity,hybrid_score,unadjusted_hybrid_score,...,withholding_reason,above_threshold,top_1_match,top_3_match,threshold_decision_match,ranking_available,match_type,outcome,expected_target_rank,top_candidates
0,Receive Request,receive request,Activity_ReceiveRequest,Receive Request,True,Activity_ReceiveRequest,Receive Request,receive request,None,None,...,None,None,True,True,None,False,direct,direct_correct,None,[]
1,Register Request,register request,Activity_RegisterRequest,Register Request,True,Activity_RegisterRequest,Register Request,register request,None,None,...,None,None,True,True,None,False,direct,direct_correct,None,[]
2,Validate Request Data,validate request data,Activity_ValidateRequestData,Validate Request Data,True,Activity_ValidateRequestData,Validate Request Data,validate request data,None,None,...,None,None,True,True,None,False,direct,direct_correct,None,[]
3,Check Supporting Documents,check supporting documents,Activity_CheckSupportingDocuments,Check Supporting Documents,True,Activity_CheckSupportingDocuments,Check Supporting Documents,check supporting documents,None,None,...,None,None,True,True,None,False,direct,direct_correct,None,[]
4,Request Additional Information,request additional information,Activity_RequestAdditionalInformation,Request Additional Information,True,Activity_RequestAdditionalInformation,Request Additional Information,request additional information,None,None,...,None,None,True,True,None,False,direct,direct_correct,None,[]
5,Assess Request,assess request,Activity_AssessRequest,Assess Request,True,Activity_AssessRequest,Assess Request,assess request,None,None,...,None,None,True,True,None,False,direct,direct_correct,None,[]
6,Approve Request,approve request,Activity_ApproveRequest,Approve Request,True,Activity_ApproveRequest,Approve Request,approve request,None,None,...,None,None,True,True,None,False,direct,direct_correct,None,[]
7,Reject Request,reject request,Activity_RejectRequest,Reject Request,True,Activity_RejectRequest,Reject Request,reject request,None,None,...,None,None,True,True,None,False,direct,direct_correct,None,[]
8,Record Decision,record decision,Activity_RecordDecision,Record Decision,True,Activity_RecordDecision,Record Decision,record decision,None,None,...,None,None,True,True,None,False,direct,direct_correct,None,[]
9,Notify Requester,notify requester,Activity_NotifyRequester,Notify Requester,True,Activity_NotifyRequester,Notify Requester,notify requester,None,None,...,None,None,True,True,None,False,direct,direct_correct,None,[]



=== v1 Incorrect Mappings ===
None

=== v1 Missing Expected Mappings ===
None

=== v1 Valid Lower-Confidence Mappings ===
None

=== v2 Direct Matches ===


...


=== v2 Incorrect Mappings ===
None

=== v2 Missing Expected Mappings ===
None

=== v2 Valid Lower-Confidence Mappings ===
None

=== v3 Direct Matches ===
None

=== v3 Incorrect Mappings ===


,source_activity,normalized_source_activity,expected_bpmn_id,expected_activity,has_valid_correspondence,suggested_bpmn_id,suggested_activity,suggested_normalized_activity,hybrid_score,unadjusted_hybrid_score,...,withholding_reason,above_threshold,top_1_match,top_3_match,threshold_decision_match,ranking_available,match_type,outcome,expected_target_rank,top_candidates
0,Log Submission,log submission,Activity_RegisterRequest,Register Request,True,Activity_RecordDecision,Record Decision,record decision,0.361217,0.361217,...,confidence-below-theta,False,False,True,False,True,candidate,incorrect_lower_confidence,2,"[{'rank': 1, 'bpmn_id': 'Activity_RecordDecisi..."
1,Finalise Request,finalise request,Activity_CloseRequest,Close Request,True,Activity_ApproveRequest,Approve Request,approve request,0.644039,0.644039,...,confidence-below-theta,False,False,True,False,True,candidate,incorrect_lower_confidence,2,"[{'rank': 1, 'bpmn_id': 'Activity_ApproveReque..."



=== v3 Missing Expected Mappings ===
None

=== v3 Valid Lower-Confidence Mappings ===


,source_activity,normalized_source_activity,expected_bpmn_id,expected_activity,has_valid_correspondence,suggested_bpmn_id,suggested_activity,suggested_normalized_activity,hybrid_score,unadjusted_hybrid_score,...,withholding_reason,above_threshold,top_1_match,top_3_match,threshold_decision_match,ranking_available,match_type,outcome,expected_target_rank,top_candidates
0,Collect Submission,collect submission,Activity_ReceiveRequest,Receive Request,True,Activity_ReceiveRequest,Receive Request,receive request,0.218474,0.368474,...,confidence-below-theta,False,True,True,False,True,candidate,correct_lower_confidence,1,"[{'rank': 1, 'bpmn_id': 'Activity_ReceiveReque..."
1,Confirm Submitted Data,confirm submitted data,Activity_ValidateRequestData,Validate Request Data,True,Activity_ValidateRequestData,Validate Request Data,validate request data,0.348009,0.498009,...,confidence-below-theta,False,True,True,False,True,candidate,correct_lower_confidence,1,"[{'rank': 1, 'bpmn_id': 'Activity_ValidateRequ..."
2,Inspect Supporting Evidence,inspect supporting evidence,Activity_CheckSupportingDocuments,Check Supporting Documents,True,Activity_CheckSupportingDocuments,Check Supporting Documents,check supporting documents,0.436175,0.586175,...,confidence-below-theta,False,True,True,False,True,candidate,correct_lower_confidence,1,"[{'rank': 1, 'bpmn_id': 'Activity_CheckSupport..."
3,Obtain Missing Details,obtain missing details,Activity_RequestAdditionalInformation,Request Additional Information,True,Activity_RequestAdditionalInformation,Request Additional Information,request additional information,0.280128,0.430128,...,confidence-below-theta,False,True,True,False,True,candidate,correct_lower_confidence,1,"[{'rank': 1, 'bpmn_id': 'Activity_RequestAddit..."
4,Evaluate Submission,evaluate submission,Activity_AssessRequest,Assess Request,True,Activity_AssessRequest,Assess Request,assess request,0.473036,0.523036,...,confidence-below-theta,False,True,True,False,True,candidate,correct_lower_confidence,1,"[{'rank': 1, 'bpmn_id': 'Activity_AssessReques..."
5,Authorise Submission,authorise submission,Activity_ApproveRequest,Approve Request,True,Activity_ApproveRequest,Approve Request,approve request,0.399608,0.399608,...,confidence-below-theta,False,True,True,False,True,candidate,correct_lower_confidence,1,"[{'rank': 1, 'bpmn_id': 'Activity_ApproveReque..."
6,Decline Submission,decline submission,Activity_RejectRequest,Reject Request,True,Activity_RejectRequest,Reject Request,reject request,0.478492,0.528492,...,confidence-below-theta,False,True,True,False,True,candidate,correct_lower_confidence,1,"[{'rank': 1, 'bpmn_id': 'Activity_RejectReques..."
7,Document Outcome,document outcome,Activity_RecordDecision,Record Decision,True,Activity_RecordDecision,Record Decision,record decision,0.394457,0.394457,...,confidence-below-theta,False,True,True,False,True,candidate,correct_lower_confidence,1,"[{'rank': 1, 'bpmn_id': 'Activity_RecordDecisi..."
8,Inform Applicant,inform applicant,Activity_NotifyRequester,Notify Requester,True,Activity_NotifyRequester,Notify Requester,notify requester,0.425380,0.475380,...,confidence-below-theta,False,True,True,False,True,candidate,correct_lower_confidence,1,"[{'rank': 1, 'bpmn_id': 'Activity_NotifyReques..."
9,File Case,file case,Activity_ArchiveCase,Archive Case,True,Activity_ArchiveCase,Archive Case,archive case,0.644888,0.644888,...,confidence-below-theta,False,True,True,False,True,candidate,correct_lower_confidence,1,"[{'rank': 1, 'bpmn_id': 'Activity_ArchiveCase'..."



=== v4 Direct Matches ===
None

=== v4 Incorrect Mappings ===
None

=== v4 Missing Expected Mappings ===
None

=== v4 Valid Lower-Confidence Mappings ===
None

=== Persisted Output Files ===
JSON: D:\Mestrado\Tese\PM4PY\platform\backend\evaluation\activity-mapping\results\2026-08-28-141910\activity-mapping-evaluation-2026-08-28-141910.json
Markdown: D:\Mestrado\Tese\PM4PY\platform\backend\evaluation\activity-mapping\results\2026-08-28-141910\activity-mapping-evaluation-2026-08-28-141910.md
